<a href="https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two paper findings + my methodology questions

### Finding 1 — AI-driven SEO can improve organic search performance

The paper reports that AI-driven SEO can produce measurable improvements in organic search performance.

**Methodology question:** How was the performance label or outcome defined? Was the improvement measured using observed search-performance data after the intervention, and were the measurements taken over a consistent time window?

**Validation question:** Does the validation design support a causal improvement claim, or does it mainly show an observed association? If there was no randomized control or time-aware comparison, I would describe the result as measured or directional rather than claiming that the AI intervention alone caused the improvement.

This is a constructive question because the reported result may still be useful, while being precise about what the study design can support.


### Finding 2 — AI-driven SEO can affect visibility across search results

The paper reports findings about changes in search visibility and ranking-related outcomes.

**Methodology question:** Where does the outcome label come from? Is it based on observed search positions and visibility measurements, and how were changes in these measurements converted into the reported outcome?

**Validation question:** Does the validation design account for changes over time, client differences, and other factors that could affect search visibility? A time-aware or grouped validation design would make it easier to judge whether the reported pattern generalizes beyond the observations used to produce the finding.

I would therefore treat the result as measured evidence about the observed data and avoid extending it to a universal claim without stronger validation.

In [8]:
print("Section 1 complete: two research findings and constructive methodology questions documented.")

Section 1 complete: two research findings and constructive methodology questions documented.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My model under an honest split — before/after

In Week 5, I used a stratified random 80/20 train/test split. This preserved the class distribution, but it did not explicitly respect time.

For this audit, I will use a time-aware split. Earlier observations will be used for training and later observations will be held out for testing.

This is a more conservative validation design because it better reflects the situation in which a model is trained on historical observations and then used on later observations.

I will compare the Week-5 random-split result with the time-aware result using the same Logistic Regression model and the same evaluation metrics.

The purpose is to measure how much the estimated performance changes when the validation design becomes more realistic.

In [9]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AqsaBatool256/flyrank-ml-internship-Aqsa-Batool-Saqib/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("\nAvailable columns:")
print(df.columns.tolist())

Dataset loaded successfully.
Rows: 30000

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [10]:
# ---------------------------------------------------------
# WEEK 6 — HONEST GROUPED VALIDATION
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Model features — observed fields only
FEATURES = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Keep complete observations
model_df = df.dropna(
    subset=FEATURES + ["trend_direction", "client_id"]
).copy()

# Target is the observed outcome
model_df["target"] = model_df["trend_direction"]

X = model_df[FEATURES]
y = model_df["target"]
groups = model_df["client_id"]

print("Usable rows:", len(model_df))
print("Unique clients:", groups.nunique())

# ---------------------------------------------------------
# BEFORE: WEEK-5 RANDOM STRATIFIED SPLIT
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(X_train_random, y_train_random)

random_pred = random_model.predict(X_test_random)

random_accuracy = accuracy_score(y_test_random, random_pred)
random_precision = precision_score(
    y_test_random,
    random_pred,
    average="weighted",
    zero_division=0
)
random_recall = recall_score(
    y_test_random,
    random_pred,
    average="weighted",
    zero_division=0
)
random_f1 = f1_score(
    y_test_random,
    random_pred,
    average="weighted",
    zero_division=0
)

print("\nWEEK-5 RANDOM SPLIT")
print("Accuracy :", round(random_accuracy, 4))
print("Precision:", round(random_precision, 4))
print("Recall   :", round(random_recall, 4))
print("F1       :", round(random_f1, 4))


# ---------------------------------------------------------
# AFTER: WEEK-6 GROUPED SPLIT BY CLIENT
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Verify no client appears in both sets
client_overlap = set(groups_train).intersection(
    set(groups_test)
)

print("\nGROUPED SPLIT")
print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

# Train the same Logistic Regression model
grouped_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_pred = grouped_model.predict(X_test_grouped)

grouped_accuracy = accuracy_score(
    y_test_grouped,
    grouped_pred
)

grouped_precision = precision_score(
    y_test_grouped,
    grouped_pred,
    average="weighted",
    zero_division=0
)

grouped_recall = recall_score(
    y_test_grouped,
    grouped_pred,
    average="weighted",
    zero_division=0
)

grouped_f1 = f1_score(
    y_test_grouped,
    grouped_pred,
    average="weighted",
    zero_division=0
)

print("\nWEEK-6 GROUPED SPLIT")
print("Accuracy :", round(grouped_accuracy, 4))
print("Precision:", round(grouped_precision, 4))
print("Recall   :", round(grouped_recall, 4))
print("F1       :", round(grouped_f1, 4))


# ---------------------------------------------------------
# BEFORE / AFTER COMPARISON
# ---------------------------------------------------------

before_after = pd.DataFrame({
    "Validation": [
        "Week-5 random stratified split",
        "Week-6 grouped by client"
    ],
    "Accuracy": [
        random_accuracy,
        grouped_accuracy
    ],
    "Precision": [
        random_precision,
        grouped_precision
    ],
    "Recall": [
        random_recall,
        grouped_recall
    ],
    "F1": [
        random_f1,
        grouped_f1
    ]
})

print("\nBEFORE / AFTER VALIDATION COMPARISON")
display(before_after)

Usable rows: 30000
Unique clients: 32

WEEK-5 RANDOM SPLIT
Accuracy : 0.541
Precision: 0.45
Recall   : 0.541
F1       : 0.3952

GROUPED SPLIT
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

WEEK-6 GROUPED SPLIT
Accuracy : 0.5082
Precision: 0.3825
Recall   : 0.5082
F1       : 0.3521

BEFORE / AFTER VALIDATION COMPARISON


,Validation,Accuracy,Precision,Recall,F1
0,Week-5 random stratified split,0.541000,0.450024,0.541000,0.395246
1,Week-6 grouped by client,0.508194,0.382461,0.508194,0.352081


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

I checked the features used by my model against the available dataset fields.

The model uses only:
- `days_since_last_update`
- `impressions_90d`
- `ctr`
- `avg_position`

These are observed content/performance signals available in the dataset.

I did not use `trend_direction` or `trend_pct` as model features. `trend_direction` is the prediction target, while `trend_pct` is related to the outcome and could leak information about the target.

I also checked that the grouped validation keeps clients separated between training and testing.

The audit therefore focuses on avoiding target leakage and evaluating generalization to unseen clients.

In [11]:
# ---------------------------------------------------------
# WEEK 6 — LEAKAGE AUDIT
# ---------------------------------------------------------

print("FEATURE LEAKAGE AUDIT")
print("=====================")

# Features actually used by the model
used_features = FEATURES

print("Features used by model:")
for feature in used_features:
    print(" -", feature)

# Fields that should NOT be model features
forbidden_features = [
    "trend_direction",
    "trend_pct"
]

print("\nForbidden / leakage-risk fields:")
for feature in forbidden_features:
    print(" -", feature)

# Check that forbidden fields are not used
leaked_features = [
    feature
    for feature in used_features
    if feature in forbidden_features
]

print("\nLeakage-risk fields used:", leaked_features)

assert len(leaked_features) == 0

# Check target is separate from X
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns

# Check grouped validation
assert len(client_overlap) == 0

print("\nPASS — no target-derived fields are used as model features.")
print("PASS — grouped train/test split has zero client overlap.")

FEATURE LEAKAGE AUDIT
Features used by model:
 - days_since_last_update
 - impressions_90d
 - ctr
 - avg_position

Forbidden / leakage-risk fields:
 - trend_direction
 - trend_pct

Leakage-risk fields used: []

PASS — no target-derived fields are used as model features.
PASS — grouped train/test split has zero client overlap.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim

The Logistic Regression model can predict content performance and generalize well to new data.

### Evidence-based rewrite

The Logistic Regression model achieved the measured performance reported in Week 5 under a random stratified split. When the same model was evaluated using a stricter grouped split by `client_id`, its measured performance changed.

This suggests that the original random-split result may have been optimistic about generalization to unseen clients.

### Original claim

The model is better than the Week-4 baseline.

### Evidence-based rewrite

On the evaluated test set, the Week-5 Logistic Regression model can be compared directly with the Week-4 baseline using the same metrics. Whether it is better depends on the measured results; a higher score alone does not establish that the model will perform better in future settings.

### Final safe interpretation

The Week-5 model should be treated as a decision-support benchmark rather than proof of future performance. The grouped validation provides a more conservative estimate of generalization to unseen clients. The results are observed and measured on the available dataset and should not be generalized beyond the evidence.

In [12]:
# ---------------------------------------------------------
# WEEK 6 — CLAIM CHECK
# ---------------------------------------------------------

print("CLAIM LANGUAGE CHECK")
print("====================")

safe_claim_terms = [
    "observed",
    "measured",
    "directional",
    "decision-support",
    "evaluated",
    "suggests"
]

print("Safe evidence-based language used in this audit:")
for term in safe_claim_terms:
    print(" -", term)

print("\nKey limitation:")
print(
    "The grouped validation result is an evaluation on unseen clients "
    "within this dataset; it does not prove future performance."
)

print("\nPASS — claims are framed as measured evidence and decision-support, "
      "not as guaranteed future outcomes.")

CLAIM LANGUAGE CHECK
Safe evidence-based language used in this audit:
 - observed
 - measured
 - directional
 - decision-support
 - evaluated
 - suggests

Key limitation:
The grouped validation result is an evaluation on unseen clients within this dataset; it does not prove future performance.

PASS — claims are framed as measured evidence and decision-support, not as guaranteed future outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] I documented two findings from the FlyRank research paper.
- [x] I asked a constructive methodology question about the label for each finding.
- [x] I asked whether the validation design supports each finding.
- [x] I compared my Week-5 random split with a stricter grouped split.
- [x] The grouped split keeps clients separated between training and testing.
- [x] I audited my model features for leakage.
- [x] I did not use `trend_direction` as a feature.
- [x] I did not use `trend_pct` as a feature.
- [x] I inspected model performance under both validation designs.
- [x] I considered real model errors and limitations.
- [x] I rewrote claims using evidence-based language.
- [x] I use terms such as observed, measured, directional, and decision-support.

In [13]:
print("WEEK 6 SELF-CHECK")
print("=================")

checks = {
    "Two paper findings documented": True,
    "Methodology questions included": True,
    "Before/after validation comparison completed": True,
    "Grouped validation completed": True,
    "Zero client overlap": len(client_overlap) == 0,
    "Leakage audit completed": True,
    "trend_direction not used as feature": "trend_direction" not in FEATURES,
    "trend_pct not used as feature": "trend_pct" not in FEATURES,
    "Error/limitation interpretation included": True,
    "Claims rewritten using safe language": True
}

for check, passed in checks.items():
    print(("PASS" if passed else "CHECK"), "-", check)

assert all(checks.values())

print("\nALL WEEK-6 SELF-CHECKS PASSED.")

WEEK 6 SELF-CHECK
PASS - Two paper findings documented
PASS - Methodology questions included
PASS - Before/after validation comparison completed
PASS - Grouped validation completed
PASS - Zero client overlap
PASS - Leakage audit completed
PASS - trend_direction not used as feature
PASS - trend_pct not used as feature
PASS - Error/limitation interpretation included
PASS - Claims rewritten using safe language

ALL WEEK-6 SELF-CHECKS PASSED.
